In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import json
from pathlib import Path
import os
import pandas as pd
from datasets import load_dataset
from modules.model import define_tokenizer
from modules.preprocess_hc3 import preprocess_hc3
from modules.preprocess_mage import preprocess_mage
from modules.tools import extract_model_name


/Users/maksi/Develop/ai-detection/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/maksi/Develop/ai-detection/.venv-1/lib/python3.13/site-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [2]:
def load_config(config_path):
    with open(config_path, "r") as f:
        return json.load(f)

In [3]:
config = load_config(Path("configs/load_data.json"))

save_folder = Path(config["preprocess_folder"])
datasets = config["datasets"]
features = config["features"]
models = config["models"]
batch_size = config["preprocess_batch_size"]

NameError: name 'Path' is not defined

In [5]:
ds_hc3 = load_dataset(datasets["hc3"])
ds_mage = load_dataset(datasets["mage"])

Using the latest cached version of the dataset since Hello-SimpleAI/HC3 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /Users/maksi/.cache/huggingface/datasets/Hello-SimpleAI___hc3/all/1.1.0/4d0ff18143b5a7e1b1e79beb540c04549d1e59d3 (last modified on Thu Oct 29 15:00:28 2026).


In [6]:
preprocess_hc3(
    ds_hc3,
    features=features,
    extract_features=True,
    save_path=save_folder /"with_features"/"hc3_dataset.parquet",
)

# preprocess_mage(
#     ds_mage,
#     features=features,
#     extract_features=True,
#     save_path=save_folder /"with_features"/"mage_dataset.parquet",
# )

extracting features: 100%|██████████| 4866/4866 [00:33<00:00, 146.26it/s]


In [7]:
hc3 = pd.read_parquet(save_folder /"with_features"/"hc3_dataset.parquet")
# mage = pd.read_parquet(save_folder /"with_features"/"mage_dataset.parquet")

In [17]:
if not os.path.exists(save_folder / "text_only"):
    os.makedirs(save_folder / "text_only")

hc3_text_only = hc3.drop(columns=features)
hc3_text_only.to_parquet(save_folder / "text_only" / "hc3_dataset.parquet")

# mage_text_only = mage.drop(columns=features)
# mage_text_only.to_parquet(save_folder / "text_only" / "mage_dataset.parquet")




for dataset in [hc3]:
    for model in models:
        df = pd.DataFrame()
        batches_tokenized = []
        tokenizer = define_tokenizer(model)
        prev_id = 0
        current_id = batch_size
        for i in range(len(dataset) // batch_size + (1 if len(dataset) % batch_size != 0 else 0)):
            batch = dataset.iloc[prev_id:current_id]

            batch = batch.reset_index(drop=True)
            batch_tokenized = tokenizer(batch["text"].tolist(), truncation=True)
            batch_tokenized = dict(batch_tokenized)
            batch_tokenized = pd.DataFrame(batch_tokenized)
            batch_tokenized["sample_id"] = batch["sample_id"]
            batch_tokenized["label"] = batch["label"]
            batch_tokenized["text"] = batch["text"]
            batch_tokenized[features] = batch[features]
            batch_tokenized["split"] = batch["split"]
            batch_tokenized["input_ids"] = batch_tokenized["input_ids"]
            batch_tokenized["attention_mask"] = batch_tokenized["attention_mask"]
            batches_tokenized.append(batch_tokenized)
            prev_id = current_id
            current_id += batch_size
        df = pd.concat(batches_tokenized)
        save_tokenized_path = save_folder / "tokenized"
        if not os.path.exists(save_tokenized_path):
            os.makedirs(save_tokenized_path)
        model_name = extract_model_name(model)
        df.to_parquet(Path(save_tokenized_path / f"{model_name}_{dataset["dataset"][1]}.parquet"))

